# Multidimensional Experimental Comparison

This output-stripped report notebook compares the current experimental multidimensional profiles for the synthetic 50D multifactor benchmark and the local S&P500 50-stock panel. It reads `trained_models/multidim_profiles.yaml`, not the public model registry. No multidimensional public default exists, and no multidimensional registry-selected public model exists.

The default execution path is metadata-only and local-output-optional: it does not train models, evaluate checkpoints, download data, or write generated artefacts.

In [1]:
USE_MULTIDIM_PROFILES = True
RUN_TRAINING = False
RUN_EVALUATION = False
ALLOW_MISSING_OUTPUTS = True
SELECT_PROFILE = "balanced_empirical"

AUTO_DISCOVER_LOCAL_SUMMARIES = False
AUTO_DISCOVER_LOCAL_BATCHES = False
PLOT_LOCAL_BATCHES = False
MAX_BATCH_FILES = 2
MAX_BATCH_FILE_BYTES = 90 * 1024 * 1024
MAX_PLOT_VALUES = 8000

MULTIDIM_PROFILE_PATH = "../../trained_models/multidim_profiles.yaml"

## Setup

Path handling resolves from the repository root so the notebook can run from `jupyter nbconvert` or an interactive kernel started inside the repository. Matplotlib writes its cache under `/tmp` to avoid touching user-level state during smoke execution.

In [2]:
import importlib.util
import json
import os
import sys
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-time-causal-vae")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
NOTEBOOK_DIR = REPO_ROOT / "notebooks" / "report"
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from time_causal_vae.experiments.multidim_profiles import (
    list_multidim_profiles,
    load_multidim_profiles,
    select_multidim_profile,
)

SCIPY_AVAILABLE = importlib.util.find_spec("scipy") is not None
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})


def resolve_repo_path(path: str | Path) -> Path:
    raw = Path(path).expanduser()
    if raw.is_absolute():
        return raw
    notebook_relative = (NOTEBOOK_DIR / raw).resolve()
    if notebook_relative.exists():
        return notebook_relative
    return (REPO_ROOT / raw).resolve()


def display_path(path: str | Path) -> str:
    resolved = Path(path).resolve()
    try:
        return str(resolved.relative_to(REPO_ROOT))
    except ValueError:
        return str(resolved)


def read_json_if_present(path: str | Path) -> dict[str, Any] | None:
    resolved = resolve_repo_path(path)
    if not resolved.exists():
        return None
    with resolved.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    return payload if isinstance(payload, dict) else {"payload_type": type(payload).__name__}


def flatten_numeric(
    mapping: Mapping[str, Any], prefix: str = "", max_depth: int = 3
) -> dict[str, float]:
    flattened: dict[str, float] = {}
    if max_depth < 0:
        return flattened
    for key, value in mapping.items():
        name = f"{prefix}.{key}" if prefix else str(key)
        if isinstance(value, bool):
            continue
        if isinstance(value, int | float | np.number):
            flattened[name] = float(value)
        elif isinstance(value, Mapping):
            flattened.update(flatten_numeric(value, name, max_depth - 1))
    return flattened


def compact_caveats(caveats: Any) -> str:
    if isinstance(caveats, list):
        return "; ".join(str(item) for item in caveats)
    return "" if caveats is None else str(caveats)


PROFILE_PATH = resolve_repo_path(MULTIDIM_PROFILE_PATH)
print(f"repo_root={REPO_ROOT}")
print(f"profile_path={display_path(PROFILE_PATH)}")
print(f"scipy_available={SCIPY_AVAILABLE}")

repo_root=/home/georgios-vourvachakis/Desktop/TimeCausalVQVAE
profile_path=trained_models/multidim_profiles.yaml
scipy_available=True


## Load Multidimensional Experimental Profiles

The profile file records experiment-level `status: experimental` and `public_default: false`. Checkpoint fields are local path conventions only.

In [3]:
if not USE_MULTIDIM_PROFILES:
    raise RuntimeError("This notebook is intended to use multidimensional experimental profiles.")

profiles = load_multidim_profiles(PROFILE_PATH)
available_profiles = list_multidim_profiles(PROFILE_PATH)

profile_rows: list[dict[str, Any]] = []
for experiment, section in profiles.items():
    for profile_name, profile_section in section.get("profiles", {}).items():
        evidence = profile_section.get("evidence", {})
        sampling = profile_section.get("sampling", {})
        profile_rows.append({
            "experiment": experiment,
            "profile": profile_name,
            "family": profile_section.get("family"),
            "candidate": profile_section.get("candidate"),
            "experiment_status": section.get("status"),
            "public_default": section.get("public_default"),
            "sampling": ", ".join(f"{key}={value}" for key, value in sampling.items()),
            "summary": evidence.get("profile_summary", ""),
            "caveats": compact_caveats(profile_section.get("caveats")),
        })

profile_table = pd.DataFrame(profile_rows)
display(profile_table)
display(Markdown("**Available profile keys:** `" + str(available_profiles) + "`"))
display(Markdown("**Registry status:** no multidimensional registry-selected public model exists."))

,experiment,profile,family,candidate,experiment_status,public_default,sampling,summary,caveats
0,multifactor_market,correlation_sector,continuous,beta_cvae_latent8_hidden64,experimental,False,,"Stronger synthetic correlation, eigenspectrum,...",Portfolio-volatility compression remains mater...
1,multifactor_market,portfolio_tail,discrete,factor_pca_rvq_q2_cb64_factorised,experimental,False,"temperature=1.0, top_k=None, selection_split=s...",Seed-robust synthetic discrete candidate for M...,"Trails BetaCVAE on correlation, eigenspectrum,..."
2,sp500_50_panel,balanced_empirical,continuous,beta_cvae_latent8_hidden64_v3,experimental,False,,Best current one-seed empirical v3 candidate a...,one_seed_local; Not seed-robust on empirical v...
3,sp500_50_panel,discrete_reporting,discrete,factor_pca_rvq_q2_cb64_factorised,experimental,False,"temperature=0.7, top_k=20, selection_split=emp...",Empirical discrete reporting variant only; cal...,pair_and_regime_miscalibration; over_dispersed...


**Available profile keys:** `{'multifactor_market': ['correlation_sector', 'portfolio_tail'], 'sp500_50_panel': ['balanced_empirical', 'discrete_reporting']}`

**Registry status:** no multidimensional registry-selected public model exists.

## Continuous Versus Discrete Candidates By Profile

The table below uses representative metrics stored in the experimental profile metadata. It is a profile-labelled comparison, not a public model selection.

In [4]:
COMPARISON_PROFILES = {
    "synthetic correlation/sector": ("multifactor_market", "correlation_sector"),
    "synthetic portfolio/tail": ("multifactor_market", "portfolio_tail"),
    "empirical balanced": ("sp500_50_panel", "balanced_empirical"),
    "empirical discrete reporting": ("sp500_50_panel", "discrete_reporting"),
}

METRIC_ALIASES = {
    "MMD": ["mmd", "nojump_mmd_mean"],
    "SWD": ["swd", "nojump_swd_mean"],
    "cov rel Frobenius": [
        "cov_rel_frobenius",
        "nojump_cov_rel_frobenius_mean",
        "jump_cov_rel_frobenius_mean",
    ],
    "corr rel Frobenius": [
        "corr_rel_frobenius",
        "nojump_corr_rel_frobenius_mean",
        "jump_corr_rel_frobenius_mean",
    ],
    "corr spectrum": ["corr_spectrum", "nojump_corr_spectrum_mean", "jump_corr_spectrum_mean"],
    "sector MAE": ["sector_mae", "nojump_sector_mae_mean", "jump_sector_mae_mean"],
    "equal-weight vol ratio": [
        "equal_weight_vol_ratio",
        "nojump_equal_weight_vol_ratio_mean",
        "jump_equal_weight_vol_ratio_mean",
    ],
    "random-portfolio vol ratio": ["random_portfolio_vol_ratio"],
    "VaR q01 error": ["equal_weight_var_q01_error", "nojump_equal_weight_var_q01_error_mean"],
    "ES q01 error": ["equal_weight_es_q01_error", "nojump_equal_weight_es_q01_error_mean"],
    "q0 active": ["q0_active_codes", "nojump_q0_active_codes"],
    "q1 active": ["q1_active_codes", "nojump_q1_active_codes"],
}


def metric_by_alias(metrics: Mapping[str, Any], aliases: Sequence[str]) -> float | None:
    for alias in aliases:
        value = metrics.get(alias)
        if isinstance(value, int | float | np.number) and not isinstance(value, bool):
            return float(value)
    return None


comparison_rows: list[dict[str, Any]] = []
for label, (experiment, profile_name) in COMPARISON_PROFILES.items():
    selected = select_multidim_profile(experiment, profile_name, path=PROFILE_PATH)
    metadata = selected.metadata
    evidence = metadata.get("evidence", {})
    metrics = evidence.get("representative_metrics", {})
    row: dict[str, Any] = {
        "comparison profile": label,
        "experiment": experiment,
        "profile": profile_name,
        "family": metadata.get("family"),
        "candidate": metadata.get("candidate"),
    }
    for metric_name, aliases in METRIC_ALIASES.items():
        row[metric_name] = metric_by_alias(metrics, aliases)
    comparison_rows.append(row)

comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table)

,comparison profile,experiment,profile,family,candidate,MMD,SWD,cov rel Frobenius,corr rel Frobenius,corr spectrum,sector MAE,equal-weight vol ratio,random-portfolio vol ratio,VaR q01 error,ES q01 error,q0 active,q1 active
0,synthetic correlation/sector,multifactor_market,correlation_sector,continuous,beta_cvae_latent8_hidden64,NaN,NaN,NaN,0.298700,0.107400,0.094200,0.82650,NaN,NaN,NaN,NaN,NaN
1,synthetic portfolio/tail,multifactor_market,portfolio_tail,discrete,factor_pca_rvq_q2_cb64_factorised,0.223600,0.002677,0.130100,0.360500,NaN,0.139100,0.95870,NaN,0.000997,0.002694,64.0,64.0
2,empirical balanced,sp500_50_panel,balanced_empirical,continuous,beta_cvae_latent8_hidden64_v3,0.413115,0.005034,0.421567,0.838303,0.236516,0.223918,1.07188,1.063057,0.000311,0.000647,NaN,NaN
3,empirical discrete reporting,sp500_50_panel,discrete_reporting,discrete,factor_pca_rvq_q2_cb64_factorised,0.296044,0.004152,1.105872,1.345099,0.414596,0.430765,1.58138,1.566198,0.018045,0.031749,63.0,54.0


## Optional Local Output Summaries

When local `outputs/` summaries exist, the notebook loads compact numeric snapshots. Missing files are expected on a clean checkout and do not fail the notebook when `ALLOW_MISSING_OUTPUTS=True`.

In [5]:
SUMMARY_CANDIDATES = (
    {
        "synthetic continuous seed robustness": "outputs/multifactor_market_continuous_seed_robustness/aggregate.json",
        "synthetic factor-RVQ seed robustness": "outputs/multifactor_factor_rvq_seed_robustness/aggregate.json",
        "synthetic sampling calibration": "outputs/multifactor_sampling_calibration/aggregate_summary.json",
        "sector-group research prior": "outputs/multifactor_sector_group_priors/evaluation/factorised_additive_q2_cb64_seed0/multifactor_sector_group_prior_summary.json",
        "empirical v3 comparison": "outputs/sp500_50_panel_v3_first_model_comparison/evaluation/sp500_50_panel_v3_model_comparison_summary.json",
    }
    if AUTO_DISCOVER_LOCAL_SUMMARIES
    else {}
)

summary_rows: list[dict[str, Any]] = []
for label, path in SUMMARY_CANDIDATES.items():
    resolved = resolve_repo_path(path)
    payload = read_json_if_present(resolved)
    row: dict[str, Any] = {
        "summary": label,
        "path": display_path(resolved),
        "present": payload is not None,
    }
    if payload is not None:
        numeric = flatten_numeric(payload)
        for key in [
            "mmd",
            "swd",
            "cov_rel_frobenius",
            "corr_rel_frobenius",
            "corr_spectrum",
            "sector_mae",
            "equal_weight_vol_ratio",
            "random_portfolio_vol_ratio",
            "eval_pair_l1",
            "eval_absent_pair_mass",
        ]:
            matching = [value for name, value in numeric.items() if name.endswith(key)]
            row[key] = matching[0] if matching else None
        row["numeric_metric_count"] = len(numeric)
    summary_rows.append(row)

summary_table = pd.DataFrame(summary_rows)
if summary_table.empty:
    summary_table = pd.DataFrame([{"status": "local output summary discovery disabled"}])
display(summary_table)
if "present" in summary_table and not summary_table["present"].any() and not ALLOW_MISSING_OUTPUTS:
    raise FileNotFoundError("No local multidimensional output summaries were found.")

,status
0,local output summary discovery disabled


## Optional Local Batch Distribution Plots

If local multidimensional tensor batches are present, this cell plots capped marginal histograms and KDE/ECDF summaries. It does not evaluate models or create files. If no compatible batches are found, the cell reports that state and continues.

In [6]:
BATCH_SEARCH_ROOTS = [
    "outputs/multifactor_factor_rvq_seed_robustness/priors",
    "outputs/multifactor_sector_group_priors/evaluation",
    "outputs/sp500_50_panel_v3_first_model_comparison/evaluation",
]
BATCH_PATTERNS = [
    "**/*prior_tensors.pt",
    "**/*evaluation_batch.pt",
    "**/*samples.pt",
    "**/*sample*.npz",
    "**/*batch*.npz",
]


def discover_batch_files() -> list[Path]:
    files: list[Path] = []
    if not AUTO_DISCOVER_LOCAL_BATCHES:
        return files
    for root in BATCH_SEARCH_ROOTS:
        resolved_root = resolve_repo_path(root)
        if not resolved_root.exists():
            continue
        for pattern in BATCH_PATTERNS:
            files.extend(path for path in resolved_root.glob(pattern) if path.is_file())
    unique = sorted(set(files), key=lambda path: (path.stat().st_size, str(path)))
    return [path for path in unique if path.stat().st_size <= MAX_BATCH_FILE_BYTES][
        :MAX_BATCH_FILES
    ]


def tensor_to_numpy(value: Any) -> np.ndarray | None:
    if isinstance(value, torch.Tensor):
        array = value.detach().cpu().float().numpy()
        return array if array.size else None
    if isinstance(value, np.ndarray):
        return value.astype(float, copy=False) if value.size else None
    return None


def first_numeric_array(payload: Any) -> np.ndarray | None:
    direct = tensor_to_numpy(payload)
    if direct is not None and direct.ndim >= 2:
        return direct
    if isinstance(payload, Mapping):
        preferred_keys = ["generated", "samples", "sampled", "real", "data", "paths", "returns"]
        ordered_values = [payload[key] for key in preferred_keys if key in payload]
        ordered_values.extend(payload.values())
        for value in ordered_values:
            array = first_numeric_array(value)
            if array is not None and array.ndim >= 2:
                return array
    if isinstance(payload, (list, tuple)):
        for value in payload:
            array = first_numeric_array(value)
            if array is not None and array.ndim >= 2:
                return array
    return None


def load_batch_array(path: Path) -> np.ndarray | None:
    try:
        if path.suffix == ".pt":
            payload = torch.load(path, map_location="cpu")
        elif path.suffix == ".npz":
            with np.load(path) as payload:
                payload = {key: payload[key] for key in payload.files}
        elif path.suffix == ".npy":
            payload = np.load(path)
        else:
            return None
    except Exception as exc:
        print(f"Skipping {display_path(path)}: {exc}")
        return None
    return first_numeric_array(payload)


def sample_flat_values(array: np.ndarray, max_values: int) -> np.ndarray:
    values = np.asarray(array, dtype=float).reshape(-1)
    values = values[np.isfinite(values)]
    if values.size > max_values:
        rng = np.random.default_rng(0)
        values = rng.choice(values, size=max_values, replace=False)
    return values


batch_files = discover_batch_files()
batch_rows: list[dict[str, Any]] = []
batch_values: dict[str, np.ndarray] = {}
for path in batch_files:
    array = load_batch_array(path)
    values = sample_flat_values(array, MAX_PLOT_VALUES) if array is not None else np.array([])
    label = display_path(path)
    batch_rows.append({
        "path": label,
        "loaded": array is not None,
        "shape": None if array is None else tuple(array.shape),
        "plotted_values": int(values.size),
    })
    if values.size:
        batch_values[label] = values

display(
    pd.DataFrame(batch_rows)
    if batch_rows
    else pd.DataFrame([{"status": "no compatible local batch files found"}])
)

if PLOT_LOCAL_BATCHES and batch_values:
    ncols = 3 if SCIPY_AVAILABLE else 2
    fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 3.5))
    axes = np.atleast_1d(axes)
    for label, values in batch_values.items():
        short_label = Path(label).name
        axes[0].hist(values, bins=60, density=True, alpha=0.35, label=short_label)
        sorted_values = np.sort(values)
        ecdf = np.linspace(0.0, 1.0, sorted_values.size, endpoint=False)
        if SCIPY_AVAILABLE:
            pd.Series(values).plot.kde(ax=axes[1], label=short_label)
            axes[2].plot(sorted_values, ecdf, label=short_label)
        else:
            axes[1].plot(sorted_values, ecdf, label=short_label)
    axes[0].set_title("Marginal histogram")
    axes[1].set_title("Marginal KDE" if SCIPY_AVAILABLE else "Marginal ECDF")
    if SCIPY_AVAILABLE:
        axes[2].set_title("Marginal ECDF")
    for axis in axes:
        axis.legend(fontsize=7)
    plt.tight_layout()
    plt.show()
elif not ALLOW_MISSING_OUTPUTS:
    raise FileNotFoundError("No compatible local batches were available for plotting.")

,status
0,no compatible local batch files found


## Optional Codebook And Token Diagnostics

Codebook summaries and token-dataset summaries are loaded when local tokenizer outputs are available. These diagnostics are descriptive only and do not update profile metadata.

In [7]:
DIAGNOSTIC_ROOTS = [
    "outputs/multifactor_factor_rvq_seed_robustness",
    "outputs/multifactor_sector_group_tokenizer_ablation",
    "outputs/sp500_50_panel_v3_first_model_comparison",
]
DIAGNOSTIC_PATTERNS = [
    "**/codebook_summary.json",
    "**/token_dataset_summary.json",
    "**/*token*summary.json",
]

diagnostic_files: list[Path] = []
if AUTO_DISCOVER_LOCAL_SUMMARIES:
    for root in DIAGNOSTIC_ROOTS:
        resolved_root = resolve_repo_path(root)
        if not resolved_root.exists():
            continue
        for pattern in DIAGNOSTIC_PATTERNS:
            diagnostic_files.extend(path for path in resolved_root.glob(pattern) if path.is_file())

diagnostic_rows: list[dict[str, Any]] = []
for path in sorted(set(diagnostic_files))[:20]:
    payload = read_json_if_present(path)
    numeric = flatten_numeric(payload or {})
    row: dict[str, Any] = {
        "path": display_path(path),
        "present": payload is not None,
        "numeric_metric_count": len(numeric),
    }
    for metric_hint in [
        "active",
        "perplexity",
        "pair_l1",
        "absent_pair_mass",
        "codebook_size",
        "num_quantizers",
    ]:
        matches = [value for key, value in numeric.items() if metric_hint in key]
        row[metric_hint] = matches[0] if matches else None
    diagnostic_rows.append(row)

display(
    pd.DataFrame(diagnostic_rows)
    if diagnostic_rows
    else pd.DataFrame([{"status": "no local token diagnostics found"}])
)

,status
0,no local token diagnostics found


## Commands And Public Status

The commands below are printed for local reproduction only. They are not executed by this notebook. Multidimensional models remain experimental profile labels and are not selected in `trained_models/model_registry.yaml`.

In [8]:
selected_empirical = select_multidim_profile("sp500_50_panel", SELECT_PROFILE, path=PROFILE_PATH)
selected_metadata = selected_empirical.metadata

commands = [
    "poetry run python scripts/smoke_multifactor_market_dataset.py --n-samples 256 --n-assets 50 --n-factors 5 --n-timesteps 60 --seed 99 --standardize-returns --output-dir outputs/multifactor_market_public_smoke",
    "poetry run python scripts/download_sp500_50_panel.py --start 2020-01-01 --end 2021-12-31 --output-root data --include-sector-etfs --condition-mode v3_prefix_market",
    f"poetry run python scripts/select_multidim_profile.py --experiment {selected_empirical.experiment} --profile {selected_empirical.profile}",
]

display(
    pd.DataFrame([
        {
            "selected_profile": SELECT_PROFILE,
            "family": selected_empirical.family,
            "candidate": selected_metadata.get("candidate"),
            "config": selected_metadata.get("config") or selected_metadata.get("prior_config"),
            "public_default": selected_empirical.public_default,
        }
    ])
)
display(Markdown("**Printed commands only:**"))
for command in commands:
    print(command)

display(
    Markdown(
        "**Decision:** no multidimensional registry-selected public model exists; use profile labels for local analysis."
    )
)

,selected_profile,family,candidate,config,public_default
0,balanced_empirical,continuous,beta_cvae_latent8_hidden64_v3,configs/experiments/sp500_50_panel_v3_beta_cva...,False


**Printed commands only:**

poetry run python scripts/smoke_multifactor_market_dataset.py --n-samples 256 --n-assets 50 --n-factors 5 --n-timesteps 60 --seed 99 --standardize-returns --output-dir outputs/multifactor_market_public_smoke
poetry run python scripts/download_sp500_50_panel.py --start 2020-01-01 --end 2021-12-31 --output-root data --include-sector-etfs --condition-mode v3_prefix_market
poetry run python scripts/select_multidim_profile.py --experiment sp500_50_panel --profile balanced_empirical


**Decision:** no multidimensional registry-selected public model exists; use profile labels for local analysis.